In [1]:
import matplotlib.pyplot as plt
import numpy as np
from learning.cva_estimator_portfolio_int import CVAEstimatorPortfolioInt
from simulation.diffusion_engine import DiffusionEngine
import time
import torch
import matplotlib as mpl
import pickle

/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/random.py:45: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit(forceobj=_forceobj, looplift=_looplift)
/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/random.py:71: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit(forceobj=_forceobj, looplift=_looplift)
/hom

In [2]:
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False # don't allow cudnn to tune for every input size
torch.backends.cudnn.enabled = True

device = torch.device('cuda:0')

num_coarse_steps = 10
dT = 0.01
num_fine_per_coarse = 1
dt = dT/num_fine_per_coarse
num_paths = 50000
num_inner_paths = 1
num_defs_per_path = 256
'''num_fine_per_coarse = 25
dt = 0.0004
num_paths = 100000
num_inner_paths = 1024
num_defs_per_path = 64'''
num_rates = 10
num_spreads = 9
R = np.eye(2*num_rates-1+num_spreads, dtype=np.float32) # we set the correlation matrix to the identity matrix, although not needed
initial_values = np.empty(2*num_rates-1+num_spreads, dtype=np.float32)
initial_defaults = np.empty((num_spreads-1+7)//8, dtype=np.int8)

# rates diffusion parameters
rates_params = np.empty(num_rates, dtype=[('a', '<f4'), ('b', '<f4'), ('sigma', '<f4')])
rates_params['a'] = np.random.normal(0.4, 0.03, num_rates)
rates_params['b'] = np.random.normal(0.03, 0.001, num_rates)
rates_params['sigma'] = np.abs(np.random.normal(0.0025, 0.00025, num_rates))
initial_values[:num_rates] = 0.01

# FX diffusion parameters
fx_params = np.empty(num_rates-1, dtype=[('vol', '<f4')])
fx_params['vol'] = np.abs(np.random.normal(0.25, 0.025, num_rates-1))
initial_values[num_rates:2*num_rates-1] = 1

# stochastic intensities diffusion parameters
spreads_params = np.empty(num_spreads, dtype=[('a', '<f4'), ('b', '<f4'), ('vvol', '<f4')])
spreads_params['a'] = np.random.normal(0.5, 0.03, num_spreads)
spreads_params['b'] = np.random.normal(0.01, 0.001, num_spreads)
spreads_params['vvol'] = np.abs(np.random.normal(0.0075, 0.00075, num_spreads))
initial_values[2*num_rates-1:] = 0.01

# initial default indicators
initial_defaults[:] = 0

# length of simulated path on the GPU (paths are then simulated by chunks of cDtoH_freq until maturity)
cDtoH_freq = 20

# product specs (DO NOT use the ZCs)
num_vanillas = 0
vanilla_specs = np.empty(num_vanillas,
                         dtype=[('maturity', '<f4'), ('notional', '<f4'),
                                ('strike', '<f4'), ('cpty', '<i4'),
                                ('undl', '<i4'), ('call_put', '<b1')])

num_irs = 500
irs_specs = np.empty(num_irs,
                     dtype=[('first_reset', '<f4'), ('reset_freq', '<f4'),
                            ('notional', '<f4'), ('swap_rate', '<f4'),
                            ('num_resets', '<i4'), ('cpty', '<i4'),
                            ('undl', '<i4')])

irs_specs['first_reset'] = 0.  # First reset date in the swaps
irs_specs['reset_freq'] = 0.2  # Reset frequency
irs_specs['notional'] = 10000. * \
    ((np.random.choice((-1, 1), num_irs, p=(0.5, 0.5)))
     * np.random.choice(range(1, 11), num_irs))  # Notional of the swaps
irs_specs['swap_rate'] = np.abs(np.random.normal(0.03, 0.001, num_irs))  # Swap rate, not needed, swaps are priced at par anyway
irs_specs['num_resets'] = np.random.randint(6, 51, num_irs, np.int32)  # Number of resets (num_resets*reset_freq should be equal to the desired maturity)
irs_specs['cpty'] = np.random.randint(0, num_spreads-1, num_irs, np.int32)  # Counterparty with which the swap was entered into
irs_specs['undl'] = np.random.randint(0, num_rates-1, num_irs, np.int32)  # Underlying currency

num_zcs = 0
zcs_specs = np.empty(num_zcs,
                     dtype=[('maturity', '<f4'), ('notional', '<f4'),
                            ('cpty', '<i4'), ('undl', '<i4')])

In [9]:
diffusion_engine = DiffusionEngine(50, 50, num_coarse_steps, dT, num_fine_per_coarse, dt,
                                   num_paths, num_inner_paths, num_defs_per_path, 
                                   num_rates, num_spreads, R, rates_params, fx_params, 
                                   spreads_params, vanilla_specs, irs_specs, zcs_specs,
                                   initial_values, initial_defaults, cDtoH_freq, device.index)

# selector for previous swap resets, need the states at those dates because of how the small non-Markovianity in the swap prices
prev_reset_arr = (np.arange(num_coarse_steps+1)-1)//2*2

/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/dispatcher.py:539: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/dispatcher.py:539: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/dispatcher.py:539: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


Successfully compiled all kernels.


In [3]:
method_list = ['linear', 'oneshot_NN', 'transfered_NN']
trained_model = {}

for name in method_list:
    with open('new_CVA0.01_{}.pickle'.format(name), 'rb') as file:
        pickle_file = pickle.load(file)
    trained_model[name] = pickle_file['model']

with open('new_CVA0.01_nested.pickle', 'rb') as file:
    nested = pickle.load(file)

In [4]:
def twin_l2_err(in_prediction, y_0, y_1):
    f = np.array(in_prediction)
    print(f, f.shape)
    norm = np.sqrt(((y_0*y_1).mean()+1e-7))
    _err_estimates_coarse_non_norm = np.sqrt(((f-y_0)**2).mean())
    _err_estimates_non_norm = np.sqrt((f**2-(y_0+y_1)*f+y_0*y_1).mean())
    _err_estimates_coarse = _err_estimates_coarse_non_norm/norm
    _err_estimates = _err_estimates_non_norm/norm
    _mean_labels = y_0.mean()
    return _err_estimates, _err_estimates_coarse, _err_estimates_non_norm, _err_estimates_coarse_non_norm

In [11]:
# nested CVA

print(twin_l2_err(nested['nested_CVA_1'].flatten(), nested['y0_CVA_1'].flatten(), nested['y1_CVA_1'].flatten()))

[1579.2601 1537.3827 1526.9651 ... 1576.7137 1588.1669 1534.4589] (12800000,)
(0.014462130649327061, 0.2630272968133994, 22.65781, 412.0847)


In [11]:
print(nested.keys())
for key in nested.keys():
    print(key, nested[key].shape)
print(nested['X_reg'][0])
print(nested['X'][:, 0])

dict_keys(['X', 'X_reg', 'nested_CVA_1', 'y0_CVA_1', 'y1_CVA_1'])
X (28, 50000)
X_reg (12800000, 45)
nested_CVA_1 (256, 50000)
y0_CVA_1 (256, 50000)
y1_CVA_1 (256, 50000)
[0.01016076 0.00979228 0.01006444 0.01052216 0.01008527 0.01009908
 0.00996533 0.01058059 0.01038608 0.00991305 0.97075385 1.0480177
 1.048456   1.0030783  1.010999   1.0006635  1.079734   1.0222102
 0.9730615  0.01002972 0.01001124 0.01006898 0.00991581 0.00998259
 0.00996838 0.01008079 0.00985642 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.        ]
[0.01016076 0.00979228 0.01006444 0.01052216 0.01008527 0.01009908
 0.00996533 0.01058059 0.01038608 0.00991305 0.97075385 1.0480177
 1.048456   1.0030783  1.010999   1.0006635  1.079734   1.0222102
 0.9730615  0.01002038 0.01002972 0.01001124 0.01006898 0.00991581
 0.00998259 0.00996838 0.01008079 0.00985642]


In [50]:
# Linear case
cva_estimator = CVAEstimatorPortfolioInt(prev_reset_arr, True, False, False, diffusion_engine, 
                                    device, 1, 2*(num_rates+num_spreads), (num_defs_per_path*num_paths)//32, 
                                    1000, 0.01, 0, reset_weights=False, linear=True, best_sol=True, no_nested_cva = True)
cva_estimator.saved_states = trained_model['linear']
cva_estimator.diffusion_engine.X[1] = nested['X']

predictor = cva_estimator.predict(as_cuda_array=True, flatten=False)
for i in range(2):
    next(predictor)
    f = torch.as_tensor(predictor.send(i), device=device).view(-1, 1)
    print(f)
prediction = f
print(twin_l2_err(prediction.flatten().cpu().numpy(), nested['y0_CVA_1'].flatten(), nested['y1_CVA_1'].flatten()))

/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/dispatcher.py:539: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


tensor([[1550.7072],
        [1550.7072],
        [1550.7072],
        ...,
        [1550.7072],
        [1550.7072],
        [1550.7072]], device='cuda:0')
tensor([[1553.4453],
        [1548.5339],
        [1537.5514],
        ...,
        [1563.8789],
        [1569.3478],
        [1536.7622]], device='cuda:0')
[1553.4453 1548.5339 1537.5514 ... 1563.8789 1569.3478 1536.7622] (12800000,)
(0.0169110872544825, 0.263181900836612, 26.494589, 412.3269)


In [51]:
# Oneshot case
cva_estimator = CVAEstimatorPortfolioInt(prev_reset_arr, True, False, False, diffusion_engine, 
                                         device, 1, 2*(num_rates+num_spreads), (num_defs_per_path*num_paths)//32, 
                                         2000, 0.01, 0, reset_weights=False, linear=False, best_sol=True, no_nested_cva = True)
cva_estimator.saved_states = trained_model['oneshot_NN']
cva_estimator.diffusion_engine.X[1] = nested['X']

predictor = cva_estimator.predict(as_cuda_array=True, flatten=False)
for i in range(2):
    next(predictor)
    f = torch.as_tensor(predictor.send(i), device=device).view(-1, 1)
    print(f)
prediction = f
print(twin_l2_err(prediction.flatten().cpu().numpy(), nested['y0_CVA_1'].flatten(), nested['y1_CVA_1'].flatten()))

/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/dispatcher.py:539: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


tensor([[1550.7072],
        [1550.7072],
        [1550.7072],
        ...,
        [1550.7072],
        [1550.7072],
        [1550.7072]], device='cuda:0')
tensor([[1540.1017],
        [1558.0778],
        [1545.9045],
        ...,
        [1639.0989],
        [1531.3490],
        [1513.4644]], device='cuda:0')
[1540.1017 1558.0778 1545.9045 ... 1639.0989 1531.349  1513.4644] (12800000,)
(0.035330277149972104, 0.26505779674835006, 55.35192, 415.26587)


In [52]:
# Transfered case
cva_estimator = CVAEstimatorPortfolioInt(prev_reset_arr, True, False, False, diffusion_engine, 
                                         device, 1, 2*(num_rates+num_spreads), (num_defs_per_path*num_paths)//32, 
                                         16, 0.01, 0, reset_weights=False, linear=False, best_sol=True, no_nested_cva = True)
cva_estimator.saved_states = trained_model['transfered_NN']
cva_estimator.diffusion_engine.X[1] = nested['X']

predictor = cva_estimator.predict(as_cuda_array=True, flatten=False)
for i in range(2):
    next(predictor)
    f = torch.as_tensor(predictor.send(i), device=device).view(-1, 1)
    print(f)
prediction = f
print(twin_l2_err(prediction.flatten().cpu().numpy(), nested['y0_CVA_1'].flatten(), nested['y1_CVA_1'].flatten()))

/home/botaoli/.conda/envs/neuralxva/lib/python3.8/site-packages/numba/cuda/dispatcher.py:539: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


tensor([[1550.7072],
        [1550.7072],
        [1550.7072],
        ...,
        [1550.7072],
        [1550.7072],
        [1550.7072]], device='cuda:0')
tensor([[1554.1381],
        [1493.5708],
        [1534.0673],
        ...,
        [1490.3267],
        [1546.1464],
        [1494.8070]], device='cuda:0')
[1554.1381 1493.5708 1534.0673 ... 1490.3267 1546.1464 1494.807 ] (12800000,)
(0.032444694931352214, 0.26449818745228426, 50.83108, 414.38913)
